# Dana’s Bakery — deterministic LangGraph orders
Run all cells in order in Google Colab. Orders contain only `item`, `quantity`, `day`, and `city`. All decisions and customer messages use Python rules and templates; no model, API key, or AI service is used.

Each invocation evaluates the same fixed inventory independently. Rugelach quantities count boxes; sourdough quantities count loaves. The graph never splits or partially fulfills an order.


In [ ]:
%pip install -q "langgraph>=1.0,<2.0"


## State and fixed business data
`total=False` lets callers omit internal fields and lets validation handle missing inputs. Runtime validation rejects invalid product, quantity, and day values, including boolean quantities. Days use the full English weekday names shown below. City is a string input; empty means pickup. Internal fields are reset for each order, with pickup and rejected as safe defaults until processing succeeds.


In [ ]:
from typing import TypedDict
from math import isclose
from pprint import pprint
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

class OrderState(TypedDict, total=False):
    item: str
    quantity: int
    day: str
    city: str
    original_item: str
    suggestion_count: int
    delivery_changed: bool
    subtotal: float
    discount: float
    status: str
    delivery: str
    total: float
    message: str
    steps: list[str]

PRICES = {"challah": 18, "babka": 45, "rugelach": 30, "sourdough": 25}
INVENTORY = {"challah": 10, "babka": 0, "rugelach": 3, "sourdough": 4}
CHAIN = ("babka", "rugelach", "sourdough", "challah")
DAYS = ("Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday")
OUTPUT_FIELDS = {"status", "item", "delivery", "total", "message", "steps"}
assert len(OrderState.__annotations__) == 14

# Returns a new list, never mutating a previous State's chronological log.
def logged(state, *entries):
    return [*state["steps"], *entries]


## Validation, stock checks, and bounded substitutions
Validation records structured findings in `steps`; rejection reasons need no extra State field. Routing reuses `check_stock` for the original product and each alternative. Each selection advances one position in the chain and increments the counter, so at most two alternatives are checked.


In [ ]:
def validate_order(state: OrderState):
    item = state.get("item", "")
    quantity = state.get("quantity", 0)
    day = state.get("day", "")
    city = state.get("city", "")
    steps = ["VALIDATE:START"]
    if not isinstance(item, str) or item not in PRICES:
        steps.append("VALIDATE:INVALID_ITEM")
    if type(quantity) is not int or quantity <= 0:
        steps.append("VALIDATE:INVALID_QUANTITY")
    if not isinstance(day, str) or day not in DAYS:
        steps.append("VALIDATE:INVALID_DAY")
    if len(steps) == 1:
        steps.append("VALIDATE:OK")
    return dict(item=item, quantity=quantity, day=day, city=city,
                original_item=item, suggestion_count=0, delivery_changed=False,
                subtotal=0.0, discount=0.0, status="rejected", delivery="pickup",
                total=0.0, message="", steps=steps)


def check_stock(state: OrderState):
    item, quantity = state["item"], state["quantity"]
    available = INVENTORY[item]
    return {"steps": logged(state,
        f"STOCK:{item}:requested={quantity}:available={available}",
        "STOCK:SUFFICIENT" if available >= quantity else "STOCK:INSUFFICIENT")}


def find_alternative(state: OrderState):
    assert state["suggestion_count"] < 2
    index = CHAIN.index(state["item"]) + 1
    assert index < len(CHAIN)
    item = CHAIN[index]
    count = state["suggestion_count"] + 1
    return {"item": item, "suggestion_count": count,
            "steps": logged(state, f"ALTERNATIVE:{count}:{item}")}


## Pricing, delivery, messages, and finalization
The 15% discount applies only to a subtotal strictly above 150 NIS on a day other than Friday. The delivery fee is local to `check_delivery`. That node marks successfully processed orders accepted before message generation. Rejection bypasses pricing and delivery, records a reason in the log, and retains the last evaluated item. Finalization verifies and preserves status.


In [ ]:
def calculate_price(state: OrderState):
    subtotal = float(PRICES[state["item"]] * state["quantity"])
    eligible = subtotal > 150 and state["day"] != "Friday"
    discount = subtotal * 0.15 if eligible else 0.0
    reason = "APPLIED_15_PERCENT" if eligible else (
        "NONE_FRIDAY" if state["day"] == "Friday" else "NONE_SUBTOTAL_AT_MOST_150")
    return {"subtotal": subtotal, "discount": discount,
            "steps": logged(state, f"PRICE:{state['item']}:subtotal={subtotal:.2f}",
                            f"DISCOUNT:{reason}:amount={discount:.2f}")}


def check_delivery(state: OrderState):
    city = state["city"]
    delivery = "delivery" if city == "Jerusalem" else "pickup"
    changed = bool(city) and city != "Jerusalem"
    delivery_fee = 20.0 if delivery == "delivery" else 0.0
    total = state["subtotal"] - state["discount"] + delivery_fee
    return {"delivery": delivery, "delivery_changed": changed, "total": total,
            "status": "accepted",
            "steps": logged(state,
                f"DELIVERY:{delivery}:fee={delivery_fee:.2f}:changed={changed}",
                "STATUS:accepted")}


def reject_order(state: OrderState):
    invalid = next((entry.split(":", 1)[1] for entry in state["steps"]
                    if entry.startswith("VALIDATE:INVALID_")), None)
    if invalid:
        reason = invalid
    elif state["suggestion_count"] >= 2:
        reason = "INSUFFICIENT_STOCK"
    else:
        assert CHAIN.index(state["item"]) == len(CHAIN) - 1
        reason = "NO_ALTERNATIVE"
    return {"status": "rejected", "total": 0.0,
            "steps": logged(state, f"REJECT:{reason}")}


def generate_message(state: OrderState):
    if state["status"] == "rejected":
        reasons = {
            "INVALID_ITEM": "that product is not on our active menu",
            "INVALID_QUANTITY": "the quantity must be a positive whole number (not a boolean)",
            "INVALID_DAY": "please provide a valid full English weekday name",
            "INSUFFICIENT_STOCK": "we cannot fill the entire quantity after checking two alternatives",
            "NO_ALTERNATIVE": "we do not have enough stock and there are no further alternatives",
        }
        reason = next(entry.split(":", 1)[1] for entry in reversed(state["steps"])
                      if entry.startswith("REJECT:"))
        message = f"We're sorry, we cannot accept your order: {reasons[reason]}. No charge will be made."
    else:
        assert state["status"] == "accepted"
        units = {"challah": "challah(s)", "babka": "babka(s)",
                 "rugelach": "box(es) of rugelach", "sourdough": "sourdough loaf/loaves"}
        message = f"Thank you for ordering from Dana's Bakery! Confirmed: {state['quantity']} {units[state['item']]}. "
        if state["item"] != state["original_item"]:
            message += f"We substituted {state['original_item']} with {state['item']} to fulfill your full quantity. "
        if state["delivery_changed"]:
            message += "Delivery is available only in Jerusalem, so your order was automatically changed to pickup. "
        elif state["delivery"] == "delivery":
            message += "Delivery to Jerusalem is included (20.00 NIS delivery fee). "
        else:
            message += "Your order is for pickup. "
        if state["discount"]:
            message += f"Your 15% baked-goods discount is {state['discount']:.2f} NIS. "
        message += f"Final price: {state['total']:.2f} NIS. We look forward to seeing you!"
    return {"message": message, "steps": logged(state, f"MESSAGE:{state['status']}")}


def finalize_order(state: OrderState):
    assert OUTPUT_FIELDS <= state.keys()
    assert state["status"] in ("accepted", "rejected")
    assert state["delivery"] in ("pickup", "delivery")
    assert state["message"]
    if state["status"] == "accepted":
        assert "STATUS:accepted" in state["steps"]
        assert not any(s.startswith("REJECT:") for s in state["steps"])
        assert INVENTORY[state["item"]] >= state["quantity"]
        fee = 20.0 if state["delivery"] == "delivery" else 0.0
        assert isclose(state["total"], state["subtotal"] - state["discount"] + fee)
    else:
        assert any(s.startswith("REJECT:") for s in state["steps"])
    return {"status": state["status"],
            "total": 0.0 if state["status"] == "rejected" else state["total"],
            "steps": logged(state, f"FINALIZE:{state['status']}")}


## Conditional routing and graph compilation
Exactly eight Nodes are registered. Rejected and accepted paths both finish with message generation and finalization.


In [ ]:
def route_validation(state: OrderState):
    return "check_stock" if "VALIDATE:OK" in state["steps"] else "reject_order"


def route_stock(state: OrderState):
    if INVENTORY[state["item"]] >= state["quantity"]:
        return "calculate_price"
    if state["suggestion_count"] < 2 and CHAIN.index(state["item"]) < len(CHAIN) - 1:
        return "find_alternative"
    return "reject_order"


builder = StateGraph(OrderState)
for node in (validate_order, check_stock, find_alternative, calculate_price,
             check_delivery, generate_message, finalize_order, reject_order):
    builder.add_node(node.__name__, node)
builder.add_edge(START, "validate_order")
builder.add_conditional_edges("validate_order", route_validation,
                             {name: name for name in ("check_stock", "reject_order")})
builder.add_conditional_edges("check_stock", route_stock,
                             {name: name for name in ("calculate_price", "find_alternative", "reject_order")})
builder.add_edge("find_alternative", "check_stock")
builder.add_edge("calculate_price", "check_delivery")
builder.add_edge("check_delivery", "generate_message")
builder.add_edge("reject_order", "generate_message")
builder.add_edge("generate_message", "finalize_order")
builder.add_edge("finalize_order", END)
graph = builder.compile()


## Graph visualization
`draw_mermaid_png()` uses the Mermaid rendering service for this diagram, not an AI service. Internet access is needed for rendering. If the renderer is unavailable, the Mermaid source is displayed and order processing/tests can still run.


In [ ]:
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    print(f"PNG rendering unavailable: {type(exc).__name__}: {exc}")
    print(graph.get_graph().draw_mermaid())


## Ten required test scenarios
Each test shows its numbered description, input, final result, short processing explanation, customer message, and one overall PASS/FAIL. All assertions run silently on success; a failed check prints its expected and actual values and raises an error. Test 2 also serves as the example order.


In [ ]:
from contextlib import contextmanager

_report_values = []

def capture_test_value(value, label):
    # Capture operands as the original assertion evaluates them, exactly once.
    _report_values.append((label, value))
    return value


def report_test_assertion(expression, passed):
    if passed:
        return
    # Values were captured by the original assertion, without reevaluation.
    print(f"Check failed: {expression}")
    for label, value in _report_values:
        short_label = label.split(" (")[0].replace("expected/comparison operand", "expected/comparison").replace("actual operand", "actual")
        print(f"  {short_label}: {value!r}")


@contextmanager
def test_scenario(name):
    print("\n" + "=" * 60)
    print(name)
    try:
        yield
    except BaseException as exc:
        print(f"FAIL — {type(exc).__name__}: {exc}")
        raise
    else:
        print("PASS")


def explain_test_steps(output):
    parts = []
    rejection_reasons = {
        "INVALID_ITEM": "the requested product is not on the active menu",
        "INVALID_QUANTITY": "quantity must be a positive integer, excluding booleans",
        "INVALID_DAY": "a valid weekday is required",
        "INSUFFICIENT_STOCK": "stock is insufficient after the two permitted alternatives",
        "NO_ALTERNATIVE": "stock is insufficient and no next alternative exists",
    }
    for step in output["steps"]:
        if step == "VALIDATE:OK":
            parts.append("Input validated.")
        elif step.startswith("STOCK:") and ":requested=" in step:
            _, item, requested, available = step.split(":")
            q, stock = int(requested.split("=")[1]), int(available.split("=")[1])
            parts.append(f"{item}: {stock} available for {q} requested ({'enough' if stock >= q else 'insufficient'}).")
        elif step.startswith("ALTERNATIVE:"):
            _, count, item = step.split(":")
            parts.append(f"Alternative {count}: try {item}.")
        elif step.startswith("DISCOUNT:APPLIED"):
            parts.append(f"15% discount applied: {output['discount']:.2f} NIS.")
        elif step.startswith("DISCOUNT:NONE_FRIDAY"):
            parts.append("No discount on Friday.")
        elif step.startswith("DISCOUNT:NONE_SUBTOTAL"):
            parts.append("No discount: baked-goods subtotal is at most 150 NIS.")
        elif step.startswith("DELIVERY:"):
            if output["delivery_changed"]:
                parts.append("Outside Jerusalem: delivery automatically changed to free pickup.")
            elif output["delivery"] == "delivery":
                parts.append("Jerusalem delivery adds 20 NIS.")
            else:
                parts.append("Pickup selected; no delivery fee.")
        elif step.startswith("REJECT:"):
            parts.append("Rejected because " + rejection_reasons[step.split(":", 1)[1]] + ".")
    return " ".join(parts)


def show_test_order(invoke, order):
    city = order['city'] or 'pickup'
    print(f"Input: item={order['item']}, quantity={order['quantity']}, day={order['day']}, city={city}")
    output = invoke(order)
    print(f"Result: {output['status']} | product={output['item']} | {output['delivery']} | {output['total']:.2f} NIS")
    print("Processing:", explain_test_steps(output))
    print("Customer message:", output['message'])
    return output

CASES = [('challah', 2, 'Thursday', 'Jerusalem', 'accepted', 'challah', 'delivery', 56, 0, 0, False, ['challah']), ('babka', 2, 'Thursday', 'Jerusalem', 'accepted', 'rugelach', 'delivery', 80, 1, 0, False, ['babka', 'rugelach']), ('rugelach', 5, 'Thursday', '', 'accepted', 'challah', 'pickup', 90, 2, 0, False, ['rugelach', 'sourdough', 'challah']), ('babka', 5, 'Thursday', 'Jerusalem', 'rejected', 'sourdough', 'pickup', 0, 2, 0, False, ['babka', 'rugelach', 'sourdough']), ('croissant', 1, 'Monday', 'Jerusalem', 'rejected', 'croissant', 'pickup', 0, 0, 0, False, []), ('challah', 0, 'Monday', 'Jerusalem', 'rejected', 'challah', 'pickup', 0, 0, 0, False, []), ('challah', 9, 'Thursday', 'Jerusalem', 'accepted', 'challah', 'delivery', 157.7, 0, 24.3, False, ['challah']), ('challah', 9, 'Friday', 'Jerusalem', 'accepted', 'challah', 'delivery', 182, 0, 0, False, ['challah']), ('challah', 2, 'Monday', 'Tel Aviv', 'accepted', 'challah', 'pickup', 36, 0, 0, True, ['challah']), ('challah', 2, 'Monday', '', 'accepted', 'challah', 'pickup', 36, 0, 0, False, ['challah'])]

def assert_result(result):
    _report_values.clear()
    try:
        assert capture_test_value(set(result), 'actual operand (set(result))') == capture_test_value(set(OrderState.__annotations__), 'expected/comparison operand (set(OrderState.__annotations__))')
    except AssertionError:
        report_test_assertion('set(result) == set(OrderState.__annotations__)', False)
        raise
    else:
        report_test_assertion('set(result) == set(OrderState.__annotations__)', True)
    _report_values.clear()
    try:
        assert capture_test_value(OUTPUT_FIELDS, 'actual operand (OUTPUT_FIELDS)') <= capture_test_value(result.keys(), 'expected/comparison operand (result.keys())')
    except AssertionError:
        report_test_assertion('OUTPUT_FIELDS <= result.keys()', False)
        raise
    else:
        report_test_assertion('OUTPUT_FIELDS <= result.keys()', True)
    _report_values.clear()
    try:
        assert capture_test_value(result['status'], "actual operand (result['status'])") in capture_test_value(('accepted', 'rejected'), "expected/comparison operand (('accepted', 'rejected'))")
    except AssertionError:
        report_test_assertion("result['status'] in ('accepted', 'rejected')", False)
        raise
    else:
        report_test_assertion("result['status'] in ('accepted', 'rejected')", True)
    _report_values.clear()
    try:
        assert capture_test_value(result['steps'][0], "actual operand (result['steps'][0])") == capture_test_value('VALIDATE:START', "expected/comparison operand ('VALIDATE:START')")
    except AssertionError:
        report_test_assertion("result['steps'][0] == 'VALIDATE:START'", False)
        raise
    else:
        report_test_assertion("result['steps'][0] == 'VALIDATE:START'", True)
    _report_values.clear()
    try:
        assert capture_test_value(result['steps'][-2:], "actual operand (result['steps'][-2:])") == capture_test_value([f"MESSAGE:{result['status']}", f"FINALIZE:{result['status']}"], 'expected/comparison operand ([f"MESSAGE:{result[\'status\']}", f"FINALIZE:{result[\'status\']}"])')
    except AssertionError:
        report_test_assertion('result[\'steps\'][-2:] == [f"MESSAGE:{result[\'status\']}", f"FINALIZE:{result[\'status\']}"]', False)
        raise
    else:
        report_test_assertion('result[\'steps\'][-2:] == [f"MESSAGE:{result[\'status\']}", f"FINALIZE:{result[\'status\']}"]', True)
    _report_values.clear()
    try:
        assert capture_test_value(len([s for s in result['steps'] if s.startswith('ALTERNATIVE:')]), "actual operand (len([s for s in result['steps'] if s.startswith('ALTERNATIVE:')]))") == capture_test_value(result['suggestion_count'], "expected/comparison operand (result['suggestion_count'])") <= capture_test_value(2, 'expected/comparison operand (2)')
    except AssertionError:
        report_test_assertion("len([s for s in result['steps'] if s.startswith('ALTERNATIVE:')]) == result['suggestion_count'] <= 2", False)
        raise
    else:
        report_test_assertion("len([s for s in result['steps'] if s.startswith('ALTERNATIVE:')]) == result['suggestion_count'] <= 2", True)
    _report_values.clear()
    try:
        assert capture_test_value('REJECT:', "actual operand ('REJECT:')") not in capture_test_value(result['message'], "expected/comparison operand (result['message'])")
    except AssertionError:
        report_test_assertion("'REJECT:' not in result['message']", False)
        raise
    else:
        report_test_assertion("'REJECT:' not in result['message']", True)
    if result['status'] == 'rejected':
        _report_values.clear()
        try:
            assert capture_test_value(result['total'], "actual operand (result['total'])") == capture_test_value(0, 'expected/comparison operand (0)') and capture_test_value(result['delivery'], "actual operand (result['delivery'])") == capture_test_value('pickup', "expected/comparison operand ('pickup')")
        except AssertionError:
            report_test_assertion("result['total'] == 0 and result['delivery'] == 'pickup'", False)
            raise
        else:
            report_test_assertion("result['total'] == 0 and result['delivery'] == 'pickup'", True)
        _report_values.clear()
        try:
            assert not any(capture_test_value([s.startswith(('PRICE:', 'DISCOUNT:', 'DELIVERY:')) for s in result['steps']], 'actual per-step matches (expected: all False)'))
        except AssertionError:
            report_test_assertion("not any((s.startswith(('PRICE:', 'DISCOUNT:', 'DELIVERY:')) for s in result['steps']))", False)
            raise
        else:
            report_test_assertion("not any((s.startswith(('PRICE:', 'DISCOUNT:', 'DELIVERY:')) for s in result['steps']))", True)
        _report_values.clear()
        try:
            assert capture_test_value('sorry', "actual operand ('sorry')") in capture_test_value(result['message'], "expected/comparison operand (result['message'])")
        except AssertionError:
            report_test_assertion("'sorry' in result['message']", False)
            raise
        else:
            report_test_assertion("'sorry' in result['message']", True)
    else:
        _report_values.clear()
        try:
            assert capture_test_value(str(result['quantity']), "actual operand (str(result['quantity']))") in capture_test_value(result['message'], "expected/comparison operand (result['message'])")
        except AssertionError:
            report_test_assertion("str(result['quantity']) in result['message']", False)
            raise
        else:
            report_test_assertion("str(result['quantity']) in result['message']", True)
        _report_values.clear()
        try:
            assert capture_test_value(f"{result['total']:.2f} NIS", 'actual operand (f"{result[\'total\']:.2f} NIS")') in capture_test_value(result['message'], "expected/comparison operand (result['message'])")
        except AssertionError:
            report_test_assertion('f"{result[\'total\']:.2f} NIS" in result[\'message\']', False)
            raise
        else:
            report_test_assertion('f"{result[\'total\']:.2f} NIS" in result[\'message\']', True)
        _report_values.clear()
        try:
            assert capture_test_value(result['delivery'], "actual operand (result['delivery'])") in capture_test_value(result['message'].lower(), "expected/comparison operand (result['message'].lower())")
        except AssertionError:
            report_test_assertion("result['delivery'] in result['message'].lower()", False)
            raise
        else:
            report_test_assertion("result['delivery'] in result['message'].lower()", True)
        _report_values.clear()
        try:
            assert capture_test_value(capture_test_value('substituted', "actual operand ('substituted')") in capture_test_value(result['message'], "expected/comparison operand (result['message'])"), 'actual operand (capture_test_value(\'substituted\', "actual operand (\'substituted\')") in capture_test_value(result[\'message\'], "expected/comparison operand (result[\'message\'])"))') == capture_test_value(capture_test_value(result['item'], "actual operand (result['item'])") != capture_test_value(result['original_item'], "expected/comparison operand (result['original_item'])"), 'expected/comparison operand (capture_test_value(result[\'item\'], "actual operand (result[\'item\'])") != capture_test_value(result[\'original_item\'], "expected/comparison operand (result[\'original_item\'])"))')
        except AssertionError:
            report_test_assertion("('substituted' in result['message']) == (result['item'] != result['original_item'])", False)
            raise
        else:
            report_test_assertion("('substituted' in result['message']) == (result['item'] != result['original_item'])", True)
        _report_values.clear()
        try:
            assert capture_test_value(capture_test_value('automatically changed to pickup', "actual operand ('automatically changed to pickup')") in capture_test_value(result['message'], "expected/comparison operand (result['message'])"), 'actual operand (capture_test_value(\'automatically changed to pickup\', "actual operand (\'automatically changed to pickup\')") in capture_test_value(result[\'message\'], "expected/comparison operand (result[\'message\'])"))') == capture_test_value(result['delivery_changed'], "expected/comparison operand (result['delivery_changed'])")
        except AssertionError:
            report_test_assertion("('automatically changed to pickup' in result['message']) == result['delivery_changed']", False)
            raise
        else:
            report_test_assertion("('automatically changed to pickup' in result['message']) == result['delivery_changed']", True)
        prefixes = [s.split(':', 1)[0] for s in result['steps']]
        _report_values.clear()
        try:
            assert capture_test_value(prefixes.index('PRICE'), "actual operand (prefixes.index('PRICE'))") < capture_test_value(prefixes.index('DISCOUNT'), "expected/comparison operand (prefixes.index('DISCOUNT'))") < capture_test_value(prefixes.index('DELIVERY'), "expected/comparison operand (prefixes.index('DELIVERY'))") < capture_test_value(prefixes.index('MESSAGE'), "expected/comparison operand (prefixes.index('MESSAGE'))")
        except AssertionError:
            report_test_assertion("prefixes.index('PRICE') < prefixes.index('DISCOUNT') < prefixes.index('DELIVERY') < prefixes.index('MESSAGE')", False)
            raise
        else:
            report_test_assertion("prefixes.index('PRICE') < prefixes.index('DISCOUNT') < prefixes.index('DELIVERY') < prefixes.index('MESSAGE')", True)
TEST_DESCRIPTIONS = ['2 challahs, Thursday, Jerusalem — accepted with delivery', '2 babkas, Thursday, Jerusalem — substitute rugelach', '5 rugelach, Thursday, pickup — two alternatives to challah', '5 babkas, Thursday, Jerusalem — reject after two alternatives', '1 croissant, Monday, Jerusalem — invalid product', '0 challahs, Monday, Jerusalem — invalid quantity', '9 challahs, Thursday, Jerusalem — 15% discount', '9 challahs, Friday, Jerusalem — no discount', '2 challahs, Monday, Tel Aviv — automatically change to pickup', '2 challahs, Monday, empty city — pickup']

for number, case in enumerate(CASES, 1):
    with test_scenario(f'Test {number}: {TEST_DESCRIPTIONS[number - 1]}'):
        item, quantity, day, city, status, sold, delivery, total, alternatives, discount, changed, trace = case
        test_report_name = f'Required scenario {number}'
        result = show_test_order(graph.invoke, dict(item=item, quantity=quantity, day=day, city=city))
        assert_result(result)
        _report_values.clear()
        try:
            assert capture_test_value((result['status'], result['item'], result['delivery']), "actual operand ((result['status'], result['item'], result['delivery']))") == capture_test_value((status, sold, delivery), 'expected/comparison operand ((status, sold, delivery))'), number
        except AssertionError:
            report_test_assertion("(result['status'], result['item'], result['delivery']) == (status, sold, delivery)", False)
            raise
        else:
            report_test_assertion("(result['status'], result['item'], result['delivery']) == (status, sold, delivery)", True)
        _report_values.clear()
        try:
            assert capture_test_value(result['quantity'], "actual operand (result['quantity'])") == capture_test_value(quantity, 'expected/comparison operand (quantity)') and capture_test_value(result['original_item'], "actual operand (result['original_item'])") == capture_test_value(item, 'expected/comparison operand (item)')
        except AssertionError:
            report_test_assertion("result['quantity'] == quantity and result['original_item'] == item", False)
            raise
        else:
            report_test_assertion("result['quantity'] == quantity and result['original_item'] == item", True)
        _report_values.clear()
        try:
            assert isclose(capture_test_value(result['total'], 'actual value'), capture_test_value(total, 'expected value'), abs_tol=1e-09), number
        except AssertionError:
            report_test_assertion("isclose(result['total'], total, abs_tol=1e-09)", False)
            raise
        else:
            report_test_assertion("isclose(result['total'], total, abs_tol=1e-09)", True)
        _report_values.clear()
        try:
            assert isclose(capture_test_value(result['discount'], 'actual value'), capture_test_value(discount, 'expected value'), abs_tol=1e-09), number
        except AssertionError:
            report_test_assertion("isclose(result['discount'], discount, abs_tol=1e-09)", False)
            raise
        else:
            report_test_assertion("isclose(result['discount'], discount, abs_tol=1e-09)", True)
        _report_values.clear()
        try:
            assert capture_test_value(result['suggestion_count'], "actual operand (result['suggestion_count'])") == capture_test_value(alternatives, 'expected/comparison operand (alternatives)')
        except AssertionError:
            report_test_assertion("result['suggestion_count'] == alternatives", False)
            raise
        else:
            report_test_assertion("result['suggestion_count'] == alternatives", True)
        _report_values.clear()
        try:
            assert capture_test_value(result['delivery_changed'], "actual operand (result['delivery_changed'])") is capture_test_value(changed, 'expected/comparison operand (changed)')
        except AssertionError:
            report_test_assertion("result['delivery_changed'] is changed", False)
            raise
        else:
            report_test_assertion("result['delivery_changed'] is changed", True)
        checked = [s.split(':')[1] for s in result['steps'] if ':requested=' in s]
        _report_values.clear()
        try:
            assert capture_test_value(checked, 'actual operand (checked)') == capture_test_value(trace, 'expected/comparison operand (trace)'), (number, checked)
        except AssertionError:
            report_test_assertion('checked == trace', False)
            raise
        else:
            report_test_assertion('checked == trace', True)


## Bonus — Interactive Order Simulator

Try an order using the same graph as the tests. Select croissant, zero, or a negative quantity to see a validation rejection. Each click starts an independent order using the fixed inventory.


In [ ]:
def show_order_simulator():
    try:
        import ipywidgets as widgets
        from IPython.display import display
    except ImportError:
        print("Interactive simulator unavailable: ipywidgets and IPython are required.")
        return

    product = widgets.Dropdown(
        options=("challah", "babka", "rugelach", "sourdough", "croissant"),
        description="Product:",
    )
    quantity = widgets.IntText(value=1, description="Quantity:")
    day = widgets.Dropdown(
        options=("Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"),
        description="Day:",
    )
    city = widgets.Dropdown(
        options=(("Jerusalem", "Jerusalem"), ("Tel Aviv", "Tel Aviv"), ("Pickup (empty city)", "")),
        description="City:",
    )
    button = widgets.Button(description="Process Order", button_style="primary")
    results = widgets.Output()

    def process_order(_button):
        with results:
            results.clear_output(wait=True)
            order = {
                "item": product.value,
                "quantity": quantity.value,
                "day": day.value,
                "city": city.value,
            }
            result = graph.invoke(order)
            print(f"Status: {result['status']}")
            print(f"Item: {result['item']}")
            print(f"Delivery: {result['delivery']}")
            print(f"Total: {result['total']:.2f} NIS")
            print(f"Message: {result['message']}")
            print("Processing steps:")
            for number, step in enumerate(result["steps"], start=1):
                print(f"  {number}. {step}")

    button.on_click(process_order)
    display(widgets.VBox([product, quantity, day, city, button, results]))


show_order_simulator()
